In [1]:
from utils import load_model
from train import trainer
from dataset import test_loader, sql_vocab, text_vocab
from pathlib import Path
from torchtext.data.metrics import bleu_score

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [3]:
output_file = Path("checkpoints/text_to_sql.ckpt")

In [4]:
model = load_model(output_file).eval()

In [5]:
pred = trainer.predict(model, test_loader)

You are using a CUDA device ('NVIDIA GeForce RTX 4060 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

In [ ]:
pred = [seq for batch in pred for seq in batch]   # type: ignore
pred_tokens = [seq.split() for seq in pred]
ref_tokens  = [sql_vocab.decode(ref).split() for ref, _ in test_loader.dataset]
score = bleu_score(pred_tokens, [[ref] for ref in ref_tokens])
score #0.0020021693462934295

0.0020021693462934295

In [ ]:
question = "What are the different schools and their nicknames ordered by their founding years"
ref = "SELECT school_name, nickname FROM schools ORDER BY founding_year"

reference  = sql_vocab.tokenizer(ref)
predicted  = sql_vocab.tokenizer(model.translate(question))

print("Question:  ", question)
print("Reference: ", reference)
print("Predicted: ", predicted)

score = bleu_score([predicted], [[reference]])
print("BLEU:      ", score)

Question:   What are the different schools and their nicknames ordered by their founding years
Reference:  ['select', 'school_name', ',', 'nickname', 'from', 'schools', 'order', 'by', 'founding_year']
Predicted:  [',', ',', ',', ',', 'order', 'by', 'order', 'by', 'order', 'by']
BLEU:       0.0


: 